# 06 — Judge Robustness

Reproduces paper **Tables 4 & 5** (cross-judge agreement on per-configuration
and per-concept best-utility, computed across the matched 88,000-response set).

Reads the `judge_gemma3_12b/`, `judge_gpt4_1_mini/`, and `judge_nova_2_lite/`
summary CSVs under `results/steering_evaluations/`.


In [ ]:
from pathlib import Path
import pandas as pd
from scipy.stats import pearsonr, spearmanr

from grace.analysis.load_results import load_summary_results

MODEL = 'google/gemma-3-27b-it'
CONCEPTS = sorted(p.stem for p in Path('concepts/gpt-5/extract').glob('*.json'))
JUDGES = {'gemma3-12b': 'judge_gemma3_12b', 'gpt-4.1-mini': 'judge_gpt4_1_mini', 'nova-2-lite': 'judge_nova_2_lite'}

In [ ]:
all_rows = []
for jname, jtag in JUDGES.items():
    for c in CONCEPTS:
        for r in load_summary_results(MODEL, c, judge_tag=jtag, method='pv'):
            r['judge'] = jname; all_rows.append(r)
df = pd.DataFrame(all_rows)
df.head()

## Table 4 — Per-configuration agreement (concept score and coherence)


In [ ]:
pivot_cs = df.pivot_table(index=['concept', 'layer', 'coef'], columns='judge', values='mean_concept_score')
pivot_co = df.pivot_table(index=['concept', 'layer', 'coef'], columns='judge', values='mean_coherence')
judges = list(JUDGES)
table4 = []
for i in range(len(judges)):
    for j in range(i + 1, len(judges)):
        a, b = judges[i], judges[j]
        if a in pivot_cs and b in pivot_cs:
            cs_p, _ = pearsonr(pivot_cs[a].dropna(), pivot_cs[b].dropna())
            cs_s, _ = spearmanr(pivot_cs[a].dropna(), pivot_cs[b].dropna())
            co_p, _ = pearsonr(pivot_co[a].dropna(), pivot_co[b].dropna())
            co_s, _ = spearmanr(pivot_co[a].dropna(), pivot_co[b].dropna())
            table4.append({'pair': f'{a} vs {b}', 'cs_r': cs_p, 'cs_rho': cs_s, 'co_r': co_p, 'co_rho': co_s})
pd.DataFrame(table4).round(2)

## Table 5 — Per-concept best-utility agreement


In [ ]:
best = df.groupby(['judge', 'concept'])['mean_utility'].max().reset_index()
wide = best.pivot(index='concept', columns='judge', values='mean_utility')
table5 = []
for i in range(len(judges)):
    for j in range(i + 1, len(judges)):
        a, b = judges[i], judges[j]
        if a in wide and b in wide:
            r, _ = pearsonr(wide[a].dropna(), wide[b].dropna())
            rho, _ = spearmanr(wide[a].dropna(), wide[b].dropna())
            table5.append({'pair': f'{a} vs {b}', 'pearson_r': r, 'spearman_rho': rho})
pd.DataFrame(table5).round(2)